In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.ensemble import (
    RandomForestClassifier,
    BaggingClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)

from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [6]:
df = pd.read_csv("Data/big_mart_sales.csv")

print("Shape:", df.shape)

display(df.head())

Shape: (8523, 12)


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [7]:
median_sales = df["Item_Outlet_Sales"].median()

df["Sales_Class"] = (
    df["Item_Outlet_Sales"] >= median_sales
).astype(int)

print("Median Sales:", median_sales)

print(
    df["Sales_Class"].value_counts()
)

Median Sales: 1794.331
Sales_Class
1    4266
0    4257
Name: count, dtype: int64


In [8]:
X = df.drop(
    columns=[
        "Item_Outlet_Sales",
        "Sales_Class"
    ]
)

y = df["Sales_Class"]

In [9]:
numeric_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_columns = X.select_dtypes(
    include=["object"]
).columns

print("Numerical Columns:")
print(list(numeric_columns))

print("\nCategorical Columns:")
print(list(categorical_columns))

Numerical Columns:
['Item_Weight', 'Item_Visibility', 'Item_MRP', 'Outlet_Establishment_Year']

Categorical Columns:
['Item_Identifier', 'Item_Fat_Content', 'Item_Type', 'Outlet_Identifier', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type']


In [10]:
numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

In [11]:
preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_columns
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_columns
    )
])

In [13]:
X_processed = preprocessor.fit_transform(X)

print(
    "Processed Shape:",
    X_processed.shape
)

Processed Shape: (8523, 1604)


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (6818, 1604)
Testing: (1705, 1604)


### Evaluation Function

In [19]:
def evaluate_model(model, X_test, y_test, name):

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        average="weighted"
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="weighted"
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted"
    )

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print(f"Accuracy : {accuracy * 100:.2f}%")
    print(f"Precision: {precision * 100:.2f}%")
    print(f"Recall   : {recall * 100:.2f}%")
    print(f"F1 Score : {f1 * 100:.2f}%")

    print("\nClassification Report:")

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[
                "Low Sales",
                "High Sales"
            ]
        )
    )

    return {
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Predictions": y_pred
    }

### Implement Bagging

In [20]:
bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(
        random_state=42
    ),
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

bagging_model.fit(
    X_train,
    y_train
)

BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42),
                  n_estimators=100, n_jobs=-1, random_state=42)

In [21]:
bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(
        random_state=42
    ),
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

bagging_model.fit(
    X_train,
    y_train
)

BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42),
                  n_estimators=100, n_jobs=-1, random_state=42)

In [22]:
random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

random_forest.fit(
    X_train,
    y_train
)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [23]:
rf_result = evaluate_model(
    random_forest,
    X_test,
    y_test,
    "Random Forest"
)


Random Forest
Accuracy : 82.05%
Precision: 82.32%
Recall   : 82.05%
F1 Score : 82.01%

Classification Report:
              precision    recall  f1-score   support

   Low Sales       0.85      0.77      0.81       852
  High Sales       0.79      0.87      0.83       853

    accuracy                           0.82      1705
   macro avg       0.82      0.82      0.82      1705
weighted avg       0.82      0.82      0.82      1705



### AdaBoost

In [24]:
adaboost_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(
        max_depth=1,
        random_state=42
    ),
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)

adaboost_model.fit(
    X_train,
    y_train
)

AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1,
                                                    random_state=42),
                   learning_rate=0.5, n_estimators=100, random_state=42)

In [25]:
adaboost_result = evaluate_model(
    adaboost_model,
    X_test,
    y_test,
    "AdaBoost"
)


AdaBoost
Accuracy : 82.17%
Precision: 82.84%
Recall   : 82.17%
F1 Score : 82.08%

Classification Report:
              precision    recall  f1-score   support

   Low Sales       0.88      0.75      0.81       852
  High Sales       0.78      0.89      0.83       853

    accuracy                           0.82      1705
   macro avg       0.83      0.82      0.82      1705
weighted avg       0.83      0.82      0.82      1705



In [26]:
gradient_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gradient_model.fit(
    X_train,
    y_train
)

GradientBoostingClassifier(random_state=42)

In [27]:
gradient_result = evaluate_model(
    gradient_model,
    X_test,
    y_test,
    "Gradient Boosting"
)


Gradient Boosting
Accuracy : 81.88%
Precision: 82.10%
Recall   : 81.88%
F1 Score : 81.84%

Classification Report:
              precision    recall  f1-score   support

   Low Sales       0.85      0.78      0.81       852
  High Sales       0.79      0.86      0.83       853

    accuracy                           0.82      1705
   macro avg       0.82      0.82      0.82      1705
weighted avg       0.82      0.82      0.82      1705



### XGBoost

In [28]:
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(
    X_train,
    y_train
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [33]:
gradient_result = evaluate_model(
    gradient_model,
    X_test,
    y_test,
    "Gradient Boosting"
)


Gradient Boosting
Accuracy : 81.88%
Precision: 82.10%
Recall   : 81.88%
F1 Score : 81.84%

Classification Report:
              precision    recall  f1-score   support

   Low Sales       0.85      0.78      0.81       852
  High Sales       0.79      0.86      0.83       853

    accuracy                           0.82      1705
   macro avg       0.82      0.82      0.82      1705
weighted avg       0.82      0.82      0.82      1705



In [34]:
xgb_result = evaluate_model(
    xgb_model,
    X_test,
    y_test,
    "XGBoost"
)


XGBoost
Accuracy : 81.94%
Precision: 82.19%
Recall   : 81.94%
F1 Score : 81.90%

Classification Report:
              precision    recall  f1-score   support

   Low Sales       0.85      0.77      0.81       852
  High Sales       0.79      0.86      0.83       853

    accuracy                           0.82      1705
   macro avg       0.82      0.82      0.82      1705
weighted avg       0.82      0.82      0.82      1705



In [48]:
xgb_result = evaluate_model(
    xgb_model,
    X_test,
    y_test,
    "XGBoost"
)


XGBoost
Accuracy : 81.94%
Precision: 82.19%
Recall   : 81.94%
F1 Score : 81.90%

Classification Report:
              precision    recall  f1-score   support

   Low Sales       0.85      0.77      0.81       852
  High Sales       0.79      0.86      0.83       853

    accuracy                           0.82      1705
   macro avg       0.82      0.82      0.82      1705
weighted avg       0.82      0.82      0.82      1705



In [49]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)

xgb_result = {
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average="weighted"),
    "Recall": recall_score(y_test, y_pred, average="weighted"),
    "F1 Score": f1_score(y_test, y_pred, average="weighted")
}

In [53]:
xgb_result = evaluate_model(
    xgb_model,
    X_test,
    y_test,
    "XGBoost"
)


XGBoost
Accuracy : 81.94%
Precision: 82.19%
Recall   : 81.94%
F1 Score : 81.90%

Classification Report:
              precision    recall  f1-score   support

   Low Sales       0.85      0.77      0.81       852
  High Sales       0.79      0.86      0.83       853

    accuracy                           0.82      1705
   macro avg       0.82      0.82      0.82      1705
weighted avg       0.82      0.82      0.82      1705



In [ ]:
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# =========================
# 1. Bagging
# =========================
bagging = BaggingClassifier(
    n_estimators=100,
    random_state=42
)

bagging.fit(X_train, y_train)

y_pred = bagging.predict(X_test)

bagging_result = {
    "Model": "Bagging",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average="weighted"),
    "Recall": recall_score(y_test, y_pred, average="weighted"),
    "F1 Score": f1_score(y_test, y_pred, average="weighted")
}


# =========================
# 2. Random Forest
# =========================
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

rf_result = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average="weighted"),
    "Recall": recall_score(y_test, y_pred, average="weighted"),
    "F1 Score": f1_score(y_test, y_pred, average="weighted")
}


# =========================
# 3. AdaBoost
# =========================
adaboost = AdaBoostClassifier(
    n_estimators=100,
    random_state=42
)

adaboost.fit(X_train, y_train)

y_pred = adaboost.predict(X_test)

adaboost_result = {
    "Model": "AdaBoost",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average="weighted"),
    "Recall": recall_score(y_test, y_pred, average="weighted"),
    "F1 Score": f1_score(y_test, y_pred, average="weighted")
}


# =========================
# 4. Gradient Boosting
# =========================
gradient = GradientBoostingClassifier(
    n_estimators=100,
    random_state=42
)

gradient.fit(X_train, y_train)

y_pred = gradient.predict(X_test)

gradient_result = {
    "Model": "Gradient Boosting",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average="weighted"),
    "Recall": recall_score(y_test, y_pred, average="weighted"),
    "F1 Score": f1_score(y_test, y_pred, average="weighted")
}

In [ ]:
bagging_result
rf_result
adaboost_result
gradient_result
xgb_result

In [ ]:
metrics = [
    "Precision",
    "Recall",
    "F1 Score"
]

x = np.arange(
    len(results_df["Model"])
)

width = 0.25

plt.figure(figsize=(12, 6))

plt.bar(
    x - width,
    results_df["Precision"],
    width,
    label="Precision"
)

plt.bar(
    x,
    results_df["Recall"],
    width,
    label="Recall"
)

plt.bar(
    x + width,
    results_df["F1 Score"],
    width,
    label="F1 Score"
)

plt.xticks(
    x,
    results_df["Model"],
    rotation=25,
    ha="right"
)

plt.ylabel("Score (%)")

plt.title(
    "Ensemble Learning Performance Comparison"
)

plt.ylim(0, 100)

plt.legend()

plt.tight_layout()

plt.show()